# flask_post_queue.py

This notebook contains a copy of `flask_post_queue.py` as a code cell for interactive inspection and testing.

In [535]:
import sqlite3
import os
import uuid
import subprocess
import json
import logging
from pathlib import Path
from datetime import datetime
from db_path import get_db_path

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

logging.debug("Libraries imported successfully")


DEBUG: Libraries imported successfully


### Get Directories
Function to get the directory paths that have been added 

 Todo:
 - [ ] Figure out better connection open/close logic 

In [536]:


def get_all_directories(db_path):
    """Return all rows from the `directories` table as a list of (guid, path)."""
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute("SELECT guid, path FROM directories")
        rows = cur.fetchall()
        conn.close()
        return rows
    except Exception as e:
        logging.debug(f"✗ Error reading directories table: {e}")
        try:
            if conn:
                conn.close()
        except:
            pass
        return []


### Directory Search
Scans for files in a given directory and yeilds the results
Todo:
- [ ] TBD

In [537]:
def find_video_files(directory_path, extensions=None):
    """Yield (directory, filename) tuples for video files under `directory_path`.

    `extensions` should be a list of extensions (with leading dot).
    If not provided a sensible default set will be used.
    """
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']
    # Normalize to lowercase for comparison
    lower_exts = {e.lower() for e in extensions}

    directory_path = os.path.expanduser(directory_path)
    if not os.path.isdir(directory_path):
        logging.debug(f'Not a directory: {directory_path}')
        return

    for root, dirs, files in os.walk(directory_path):
        for file_name in files:
            _, ext = os.path.splitext(file_name)
            if ext.lower() in lower_exts:
                # Yield directory path (root) and filename separately
                yield root, file_name



### File Queue Check
Check to see if that file is already in the queue

In [538]:
def is_file_unpulled_in_queue(input_file_name: str, directory_path: str, db_path: str) -> bool:
    conn = sqlite3.connect(db_path)
    try:
        cur = conn.cursor()
        cur.execute(
            "SELECT 1 FROM queue WHERE input_file_name = ? AND directory_path = ? AND datetime_encoded IS NULL LIMIT 1",
            (input_file_name, directory_path),
        )
        return cur.fetchone() is not None
    finally:
        conn.close()


### FFProbe Function
Todo:
- [ ] Research expanding the entries on the ffprobe string for HDR and other criteria

In [539]:
def run_ffprobe(directory_path, filename):
    """Run ffprobe for a file.

    Backwards-compatible: if `filename` is None, `path_or_dir` is treated as a full file path.
    Otherwise `path_or_dir` is a directory and `filename` is joined to it."""
    try:
        file_path = os.path.join(directory_path, filename)

        cmd = [
            'ffprobe',
            '-loglevel', 'quiet',
            '-show_entries', 'format:stream=index,stream,codec_type,codec_name,channel_layout,format=nb_streams',  
            '-of', 'json',
            file_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            return {'error': f'ffprobe failed: {result.stderr}'}
        
        probe_data = json.loads(result.stdout)
        
        # Pretty-print the ffprobe JSON output
        logging.debug(json.dumps(probe_data, indent=2))
        
        return probe_data
    
    except FileNotFoundError:
        return {'error': 'ffprobe not found. Ensure ffmpeg is installed and in PATH.'}
    except subprocess.TimeoutExpired:
        return {'error': 'ffprobe timeout (file too large or network issue)'}
    except json.JSONDecodeError:
        return {'error': 'Invalid ffprobe JSON output'}
    except Exception as e:
        return {'error': str(e)}


### Check Codecs

Loops through the streams in stream_info from requires_encoding, then calls 
functions to determine if the steam needs encoding based on stream type conditions 

Todo:
- [x] Copy over the stream looping function from Boilest v1.0
- [ ] Research SVT-AV1 best practices for various media types
- [ ] Store SVT-AV1 best practice presets in the DB
- [ ] Call best-practive presets in check_video_stream
- [ ] Determine what audio codec to go with
- [ ] Determine what the compromises will be if ASS subtitles are re-encoded as SubRip
- [ ] Determine if there are consequences for deleting attachments 

In [540]:
def check_codecs(encoding_decision,stream_info, ffmpeg_command):
    streams_count = stream_info['format']['nb_streams']
    
    for i in range (0,streams_count):
        codec_type = stream_info['streams'][i]['codec_type'] 
        if codec_type == 'video':
            logging.debug('Stream ' + str(i) + ' is video')
            encoding_decision, ffmpeg_command = check_video_stream(encoding_decision, i, stream_info, ffmpeg_command)
        elif codec_type == 'audio':
            encoding_decision, ffmpeg_command = check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command)
            logging.debug('audio stream')
        elif codec_type == 'subtitle':
            encoding_decision, ffmpeg_command = check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command)
            logging.debug('subtitle stream')
        elif codec_type == 'attachment':
            encoding_decision, ffmpeg_command = check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command) 
            logging.debug('attachment stream')    
    logging.debug(encoding_decision)   
    logging.debug(ffmpeg_command)
    return encoding_decision, ffmpeg_command


In [541]:
def check_video_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the video stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    desired_video_codec = 'av1'
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    if codec_name == desired_video_codec:
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name == 'mjpeg':
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name != desired_video_codec: 
        encoding_decision = True
        svt_av1_string = "libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15"
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v ' + svt_av1_string
    else:
        logging.debug('ignoring for now')
    return encoding_decision, ffmpeg_command


In [542]:
def check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the audio stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_audio_codec = 'aac'
    #if codec_name != desired_video_codec:
    #    encoding_decision = True
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:a copy'
    return encoding_decision, ffmpeg_command


In [543]:
def check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the subtitle stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_subtitle_codec = 'srt'
    #if codec_name != desired_subtitle_codec:
    #    encoding_decision = True
    logging.debug('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:s copy'
    return encoding_decision, ffmpeg_command


In [544]:
def check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the attachment stream from check_codecs to determine if the stream needs encoding
    # This will be populated at a later date
    #desired_attachment_codec = '???'
    #if codec_name != desired_attachment_codec:
    #    encoding_decision = True
    # Note, attachments may not have a codec name if the attachment is an image
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:t copy'
    return encoding_decision, ffmpeg_command


### File Output Naming Function

In [545]:
def output_file_name_fx(input_file_name, encoding_decision):
    # Get the current extension from the filename
    name_without_ext = os.path.splitext(input_file_name)[0]
    current_ext = os.path.splitext(input_file_name)[1]
    
    # Change extension to .mkv if it's not already
    if current_ext.lower() != '.mkv':
        output_file_name = name_without_ext + '.mkv'
        encoding_decision = True
    else:
        output_file_name = input_file_name
    
    # Return just the new filename without any directory path
    return output_file_name, encoding_decision

### Filesize Hash
Used as a quick/easy hash to validate the file is the expected.  

In [546]:
def get_file_size_kb(directory_path, filename):
    try:
        file_path = os.path.join(directory_path, filename)
        file_size_bytes = Path(file_path).stat().st_size
        file_size_kb = int(file_size_bytes / 1024)
        return file_size_kb
    except FileNotFoundError:
        logging.debug(f"✗ File not found: {file_path}")
        return 0
    except Exception as e:
        logging.debug(f"✗ Error getting file size: {e}")
        return 0


### Write to Queue table


In [547]:
def write_to_queue(directory_guid, directory_path, input_file_name, output_file_name, before_file_size, ffmpeg_string, db_path):
    """Write a row into `queue` using the updated schema.

    Schema columns inserted:
      directory_guid, file_guid, directory_path, input_file_name,
      output_file_name, before_file_size, after_file_size, ffmpeg_string,
      datetime_added, datetime_pulled, datetime_encoded

    Parameters:
      - directory_guid (str)
      - directory_path (str)
      - input_file_name (str)
      - output_file_name (str)
      - before_file_size (int)
      - ffmpeg_string (str)
      - after_file_size (int|None) optional
      - db_path (str|None) optional DB path; falls back to global `db_path` variable
    """
    try:
        file_guid = str(uuid.uuid4())
        datetime_added = datetime.now().isoformat()
        after_file_size = None
        datetime_pulled = None
        datetime_encoded = None

        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO queue (directory_guid, file_guid, directory_path, input_file_name, output_file_name, before_file_size, after_file_size, ffmpeg_string, datetime_added, datetime_pulled, datetime_encoded) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (
                directory_guid,
                file_guid,
                directory_path,
                input_file_name,
                output_file_name,
                before_file_size,
                after_file_size,
                ffmpeg_string,
                datetime_added,
                datetime_pulled,
                datetime_encoded,
            ),
        )
        conn.commit()
        conn.close()

        logging.debug(f"✓ Wrote queue entry {file_guid} for {input_file_name}")
        return file_guid

    except Exception as e:
        logging.debug(f"✗ Error writing to queue: {e}")
        try:
            conn.close()
        except Exception:
            pass
        return None


### Pulling it all together

In [548]:
def run_queue_workflow(db_path=None, extensions=None):

    db_path = get_db_path()

    print(db_path)

    directories = get_all_directories(db_path)
    
    for directory_guid, directory_path in directories:
        print(f"\nScanning directory {path} (guid={guid})")
        for directory, input_file_name in find_video_files(directory_path):
            file_path = os.path.join(directory, input_file_name)
            print(f"  Found: {file_path}")
            if is_file_unpulled_in_queue(input_file_name, directory, db_path) == False:
                print(f"    Adding to queue: {input_file_name}")
                default_encoding_decision = False
                ffmpeg_command = ''
                probe_data = run_ffprobe(directory,input_file_name)
                #print(probe_data)
                output_file_name, file_encoding_decision = output_file_name_fx(input_file_name, default_encoding_decision)
                final_encoding_decision, ffmpeg_command = check_codecs(file_encoding_decision, probe_data, ffmpeg_command)
                
                print (final_encoding_decision)
                print (ffmpeg_command)
                print (output_file_name)

                if final_encoding_decision == True:
                    before_file_size = get_file_size_kb(directory,input_file_name)
                    file_guid = write_to_queue(directory_guid, directory, input_file_name, output_file_name, before_file_size, ffmpeg_command, db_path)
                    print(f"    Queued file_guid: {file_guid}")
            else:
                print(f"    Skiping: {input_file_name}")


            # Process the file here if needed


run_queue_workflow()


DEBUG: {
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "hevc",
      "codec_type": "video"
    },
    {
      "index": 1,
      "codec_name": "aac",
      "codec_type": "audio",
      "channel_layout": "stereo"
    },
    {
      "index": 2,
      "codec_name": "subrip",
      "codec_type": "subtitle"
    }
  ],
  "format": {
    "filename": "/Boil/Media/Anime\\test.mkv",
    "nb_streams": 3,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "matroska,webm",
    "format_long_name": "Matroska / WebM",
    "start_time": "0.000000",
    "duration": "452.952000",
    "size": "395300778",
    "bit_rate": "6981768",
    "probe_score": 100,
    "tags": {
      "encoder": "libebml v1.4.4 + libmatroska v1.7.1",
      "creation_time": "2023-09-07T11:39:24.000000Z"
    }
  }
}
DEBUG: Stream 0 is video
DEBUG: Steam 0 codec is: hevc
DEBUG: Steam 1 codec is: aac
DEBUG: audio stream
DEBUG: Steam 2 codec is: subrip
DEBUG: subtitle

boilest.db

Scanning directory /Boil/Media/Movies (guid=1a2c2141-cc92-43a2-8c41-9ce95f0864dd)
  Found: /Boil/Media/Anime\test.mkv
    Adding to queue: test.mkv
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15 -map 0:1 -c:a copy -map 0:2 -c:s copy
test.mkv
    Queued file_guid: 2637e163-9942-4f67-b54b-280abbec8bd3
  Found: /Boil/Media/Anime\test2.mp4
    Adding to queue: test2.mp4
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
test2.mkv
    Queued file_guid: a511247c-be7d-4f21-92c6-bb27385bb577
  Found: /Boil/Media/Anime\test_file_01.mp4
    Adding to queue: test_file_01.mp4
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
test_file_01.mkv
    Queue

DEBUG: {
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "av1",
      "codec_type": "video"
    }
  ],
  "format": {
    "filename": "/Boil/Media/TV\\out.mkv",
    "nb_streams": 1,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "matroska,webm",
    "format_long_name": "Matroska / WebM",
    "start_time": "0.017000",
    "duration": "452.867000",
    "size": "8650752",
    "bit_rate": "152817",
    "probe_score": 100,
    "tags": {
      "ENCODER": "Lavf62.0.100"
    }
  }
}
DEBUG: Stream 0 is video
DEBUG: Steam 0 codec is: av1
DEBUG: False
DEBUG:  -map 0:0 -c:v copy
DEBUG: {
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "av1",
      "codec_type": "video"
    },
    {
      "index": 1,
      "codec_name": "aac",
      "codec_type": "audio",
      "channel_layout": "stereo"
    },
    {
      "index": 2,
      "codec_name": "subrip",
      "codec_type": "subtitle

False
 -map 0:0 -c:v copy
out.mkv
  Found: /Boil/Media/TV\outputav1.mkv
    Adding to queue: outputav1.mkv
False
 -map 0:0 -c:v copy -map 0:1 -c:a copy -map 0:2 -c:s copy
outputav1.mkv
  Found: /Boil/Media/TV\test_file_04.mp4
    Adding to queue: test_file_04.mp4
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
test_file_04.mkv
    Queued file_guid: b567cd89-e95b-40c5-ab5f-8e0e25f51944
  Found: /Boil/Media/TV\test_file_05.mp4
    Adding to queue: test_file_05.mp4
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
test_file_05.mkv
    Queued file_guid: cbc76017-8d4e-4c4a-8d86-d53763ef1c98

Scanning directory /Boil/Media/Movies (guid=1a2c2141-cc92-43a2-8c41-9ce95f0864dd)
  Found: /Boil/Media/Movies\test.mkv
    Adding to queue: test.mkv
False
 -map 0:0 -c:v copy
tes

DEBUG: {
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_type": "video"
    }
  ],
  "format": {
    "filename": "/Boil/Media/Movies\\Media A\\test_file_03.mp4",
    "nb_streams": 1,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "mov,mp4,m4a,3gp,3g2,mj2",
    "format_long_name": "QuickTime / MOV",
    "start_time": "0.000000",
    "duration": "4.254250",
    "size": "775034",
    "bit_rate": "1457430",
    "probe_score": 100,
    "tags": {
      "major_brand": "isom",
      "minor_version": "512",
      "compatible_brands": "isomiso2avc1mp41",
      "encoder": "Lavf58.76.100"
    }
  }
}
DEBUG: Stream 0 is video
DEBUG: Steam 0 codec is: h264
DEBUG: True
DEBUG:  -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
DEBUG: ✓ Wrote queue entry 79e7724d-0d7e-4b9a-a828-e39d870c1213 for test_file_

    Queued file_guid: 9e0e0e83-d246-47a8-b1ea-aa116676b4cc
  Found: /Boil/Media/Movies\Media A\test_file_03.mp4
    Adding to queue: test_file_03.mp4
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
test_file_03.mkv
    Queued file_guid: 79e7724d-0d7e-4b9a-a828-e39d870c1213
